# Import libraries and download the Gwembe Council webpage

This cell prepares the Python libraries needed to work with the webpage and then downloads the Gwembe Town Council webpage.

In [28]:
import pandas as pd
import requests
import urllib3

urllib3.disable_warnings(
    urllib3.exceptions.InsecureRequestWarning
)

url = "https://www.gwembecouncil.gov.zm/?page_id=3351c"

response = requests.get(url, verify=False)

response.encoding = "cp1252"

with open("gwembe_page.html", "w", encoding="utf-8") as file:
    file.write(response.text)

print("Webpage saved successfully.")

Webpage saved successfully.


# Find PDF links on the saved webpage

This cell reads the saved HTML webpage and searches it for links pointing to PDF files.

In [29]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Read the saved webpage
with open("gwembe_page.html", "r", encoding="utf-8") as file:
    html = file.read()

soup = BeautifulSoup(html, "html.parser")

# Find PDF links
pdf_links = []

for link in soup.find_all("a", href=True):
    href = link["href"]

    if ".pdf" in href.lower():
        pdf_links.append(urljoin(
            "https://www.gwembecouncil.gov.zm/?page_id=3351",
            href
        ))

print("PDF links found:")

for i, link in enumerate(pdf_links):
    print(i, link)

PDF links found:
0 http://www.gwembecouncil.gov.zm/wp-content/uploads/2025/04/List-of-2025-Recommended-Projects-by-CDFC.pdf


# Download the PDF document

This cell downloads the first PDF found on the Gwembe Council webpage and saves it to the computer.

In [30]:
import requests

pdf_url = pdf_links[0]

pdf_response = requests.get(
    pdf_url,
    verify=False
)

with open("gwembe_document.pdf", "wb") as file:
    file.write(pdf_response.content)

print("PDF downloaded successfully.")

PDF downloaded successfully.


# Display the PDF links found

This cell displays the PDF links that were discovered in the previous HTML-processing step.

In [31]:
print("PDF links found:")
for i, link in enumerate(pdf_links):
    print(i, link)

PDF links found:
0 http://www.gwembecouncil.gov.zm/wp-content/uploads/2025/04/List-of-2025-Recommended-Projects-by-CDFC.pdf


# Install pdfplumber

This cell installs the pdfplumber Python package.

In [32]:
!pip install pdfplumber


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Extract tables from the PDF

The downloaded pdf has multiple tables in it, this code broke extracts those individual tables from each page.

In [33]:
import pdfplumber
import pandas as pd

pdf_file = "gwembe_recommended_projects_2025.pdf"

all_tables = []

table_settings = {
    "vertical_strategy": "lines",
    "horizontal_strategy": "lines",
    "snap_tolerance": 7,
    "join_tolerance": 7,
    "edge_min_length": 3
}

with pdfplumber.open(pdf_file) as pdf:

    for page_number, page in enumerate(pdf.pages, start=1):

        tables = page.extract_tables(
            table_settings=table_settings
        )

        print("Page", page_number, ":", len(tables), "table(s) found")

        for table in tables:

            if table:
                all_tables.append(table)

print("\nTotal tables extracted:", len(all_tables))

Page 1 : 1 table(s) found
Page 2 : 1 table(s) found
Page 3 : 1 table(s) found
Page 4 : 1 table(s) found
Page 5 : 1 table(s) found
Page 6 : 1 table(s) found
Page 7 : 1 table(s) found
Page 8 : 1 table(s) found
Page 9 : 1 table(s) found
Page 10 : 1 table(s) found
Page 11 : 1 table(s) found
Page 12 : 1 table(s) found
Page 13 : 1 table(s) found
Page 14 : 1 table(s) found
Page 15 : 1 table(s) found
Page 16 : 2 table(s) found
Page 17 : 1 table(s) found
Page 18 : 1 table(s) found
Page 19 : 1 table(s) found
Page 20 : 1 table(s) found
Page 21 : 1 table(s) found
Page 22 : 1 table(s) found
Page 23 : 1 table(s) found
Page 24 : 1 table(s) found
Page 25 : 1 table(s) found
Page 26 : 1 table(s) found
Page 27 : 1 table(s) found
Page 28 : 1 table(s) found
Page 29 : 1 table(s) found
Page 30 : 1 table(s) found
Page 31 : 1 table(s) found
Page 32 : 1 table(s) found
Page 33 : 1 table(s) found
Page 34 : 1 table(s) found
Page 35 : 1 table(s) found

Total tables extracted: 36


# Inspect the extracted tables

This cell checks the structure of the tables stored in all_tables before converting them into DataFrames and CSV files.

In [34]:
for i, table in enumerate(all_tables, start=1):

    print("\n==============================")
    print("TABLE", i)
    print("Rows:", len(table))
    print("Columns:", max(len(row) for row in table))

    for row in table[:3]:
        print(row)


TABLE 1
Rows: 8
Columns: 8
['S/N\n1.', 'Project Name\nConstruction of a Health Post &\nAblution Block & in Mabula', 'Project Description\nConstruction of a\nHealth Post, and\nAblution', 'Sector\nHealth', 'Project\nConstruction', 'Ward\nKkota-\nKkota', 'Zone\nMabula', 'Location\nMabula']
['2.', 'Construction of a 1X3 Classroom\nBlock at Bbondo Secondary\nSchool', 'Construction of a 1X3\nClassroom Block', 'Education', 'Construction', 'Bbondo', 'Bbondo', 'Bbondo']
['3.', 'Construction of a Health Post &\na Maternity Annex in Jongola', 'Construction of a\nHealth Post &\nMaternity Annex', 'Health', 'Construction', 'Jongola', 'Jongola', 'Jongola Area']

TABLE 2
Rows: 7
Columns: 10
['No.', 'Name of Group/ Name of\nBusiness/Company', '', '', '', '', '', '', '', '']
[None, None, 'Ward', 'Zone', 'Contact Person\n(Name)', 'Contact\nPerson\n(Phone No.)', 'Type of\nBusiness', 'Sector', 'Loan Amount\nRequested', 'Remarks']
['1', 'MACHEYA HABUNGA GENERAL\nDEALERS', 'KKOLE', 'KKOLE', 'Gilbert Mucheya

# Create functions for cleaning and combining tables

This cell defines two reusable functions that will be used to prepare the extracted tables.

In [35]:
import pandas as pd
import re

# --------------------------------
# Function to clean cell values
# --------------------------------

def clean_cell(value):

    if value is None:
        return None

    value = str(value)

    # Replace line breaks and multiple spaces
    value = " ".join(value.split())

    return value.strip()


# --------------------------------
# Function to combine tables
# --------------------------------

def combine_tables(tables, columns):

    rows = []

    for table in tables:

        for row in table:

            # Clean every cell
            row = [clean_cell(value) for value in row]

            # Make sure every row has the correct number of columns
            if len(row) < len(columns):
                row += [None] * (len(columns) - len(row))

            elif len(row) > len(columns):
                row = row[:len(columns)]

            rows.append(row)

    df = pd.DataFrame(rows, columns=columns)

    return df

# Combine the extracted PDF pages into five datasets

This cell organizes the page-level tables into the five logical datasets contained in the PDF.

In [36]:
# 1. COMMUNITY PROJECTS
community_columns = [
    "S/N",
    "Project Name",
    "Project Description",
    "Sector",
    "Project",
    "Ward",
    "Zone",
    "Location"
]

community_df = combine_tables(
    all_tables[0:1],
    community_columns
)


# 2. LOAN APPLICANTS

loan_columns = [
    "No.",
    "Name of Group/Name of Business/Company",
    "Ward",
    "Zone",
    "Contact Person (Name)",
    "Contact Person (Phone No.)",
    "Type of Business",
    "Sector",
    "Loan Amount Requested",
    "Remarks"
]

loan_df = combine_tables(
    all_tables[1:4],
    loan_columns
)



# 3. EMPOWERMENT GRANTS

empowerment_columns = [
    "S/N",
    "Name Of Applicant",
    "Ward",
    "Project Title",
    "Approved Amount"
]

empowerment_df = combine_tables(
    all_tables[4:10],
    empowerment_columns
)



# 4. SCHOOL BURSARIES
school_columns = [
    "S/N", 
    "Name of Pupil",
    "Ward",
    "Sex (M/F)",
    "Grade",
    "Name of School",
    "School Location (District)",
    "Termly Fees",
    "Annual Fees"
]

school_df = combine_tables(
    all_tables[10:16],
    school_columns
)


# 5. SKILLS DEVELOPMENT BURSARIES
skills_columns = [
    "S/N",
    "Name of Student",
    "Ward",
    "Sex (M/F)",
    "Date of Birth",
    "Name of Skill/Programme",
    "Level of Skill",
    "Programme Duration (No. of Months)",
    "Training Institute (TEVET/ZNS)",
    "Term One",
    "Term Two",
    "Term Three",
    "Annual Tuition Fees"
]

skills_df = combine_tables(
    all_tables[16:36],
    skills_columns
)


print("Tables combined successfully.")

Tables combined successfully.


# Check the size of each dataset

This cell checks the dimensions of the five DataFrames created in the previous cell.

In [37]:
print("Community Projects:", community_df.shape)
print("Loan Applicants:", loan_df.shape)
print("Empowerment Grants:", empowerment_df.shape)
print("School Bursaries:", school_df.shape)
print("Skills Development Bursaries:", skills_df.shape)

Community Projects: (8, 8)
Loan Applicants: (22, 10)
Empowerment Grants: (97, 5)
School Bursaries: (50, 9)
Skills Development Bursaries: (275, 13)


# Save the DataFrames as CSV files

This cell exports the five pandas DataFrames to CSV files.

In [38]:
community_df.to_csv(
    "community_projects.csv",
    sep="|",
    index=False
)

loan_df.to_csv(
    "loan_applicants.csv",
    sep="|",
    index=False
)

empowerment_df.to_csv(
    "empowerment_grants.csv",
    sep="|",
    index=False
)

school_df.to_csv(
    "school_bursaries.csv",
    sep="|",
    index=False
)

skills_df.to_csv(
    "skills_development_bursaries.csv",
    sep="|",
    index=False
)

print("All CSV files saved successfully.")

All CSV files saved successfully.


# Cleaning Community Projects CSV file

## Inspecting the Community Projects CSV file

Inspecting the csv file, so as to set ground for cleaning

In [39]:
import pandas as pd

In [40]:
community_projects = pd.read_csv("Community_Projects.csv", sep="|")
community_projects.shape

(8, 8)

In [41]:
community_projects[community_projects.isna().any(axis=1)]

,S/N,Project Name,Project Description,Sector,Project,Ward,Zone,Location
5,6.,Rural Electrification,Electrification of Kalama Primary School,Power Supply,Construction,Jumbo - Sompani Ward,NaN,Kalama Primary School
7,"TOTAL K18,304,493.20",NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
community_projects

,S/N,Project Name,Project Description,Sector,Project,Ward,Zone,Location
0,S/N 1.,Project Name Construction of a Health Post & A...,Project Description Construction of a Health P...,Sector Health,Project Construction,Ward Kkota- Kkota,Zone Mabula,Location Mabula
1,2.,Construction of a 1X3 Classroom Block at Bbond...,Construction of a 1X3 Classroom Block,Education,Construction,Bbondo,Bbondo,Bbondo
2,3.,Construction of a Health Post & a Maternity An...,Construction of a Health Post & Maternity Annex,Health,Construction,Jongola,Jongola,Jongola Area
3,4.,Construction of a 1X3 Classroom Block at Mukon...,Construction of a 1X3 Classroom Block,Education,Construction,Chaamwe,Mukono,Mukuno Community School
4,5.,Maintenance of Roads in the District,Maintenance of Roads & Servicing of Equipment,Road Infrastructure,Construction,Gwembe District,Gwembe District,Gwembe District
5,6.,Rural Electrification,Electrification of Kalama Primary School,Power Supply,Construction,Jumbo - Sompani Ward,NaN,Kalama Primary School
6,7.,Procurement of a Drilling Rig & a Siting Machine,Procurement of a Drilling Rig & a Siting Machine,Water,Procurement,Gwembe District,Gwembe District,Gwembe District
7,"TOTAL K18,304,493.20",NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Community Projects Cleaning

In [43]:
community_projects = community_projects.drop("S/N", axis=1)
community_projects = community_projects.drop(index=7)

In [44]:
community_projects = community_projects.apply(lambda col: col.str.lower())
community_projects

,Project Name,Project Description,Sector,Project,Ward,Zone,Location
0,project name construction of a health post & a...,project description construction of a health p...,sector health,project construction,ward kkota- kkota,zone mabula,location mabula
1,construction of a 1x3 classroom block at bbond...,construction of a 1x3 classroom block,education,construction,bbondo,bbondo,bbondo
2,construction of a health post & a maternity an...,construction of a health post & maternity annex,health,construction,jongola,jongola,jongola area
3,construction of a 1x3 classroom block at mukon...,construction of a 1x3 classroom block,education,construction,chaamwe,mukono,mukuno community school
4,maintenance of roads in the district,maintenance of roads & servicing of equipment,road infrastructure,construction,gwembe district,gwembe district,gwembe district
5,rural electrification,electrification of kalama primary school,power supply,construction,jumbo - sompani ward,NaN,kalama primary school
6,procurement of a drilling rig & a siting machine,procurement of a drilling rig & a siting machine,water,procurement,gwembe district,gwembe district,gwembe district


## Saving cleaned community_projects to csv

In [45]:
community_projects.to_csv(
    "cleaned_community_projects.csv",
    sep="|",
    index=False
)

# Cleaning Loan Applicants CSV file

## Inspecting the Loan Applicants CSV file

Inspecting the csv file, so as to set ground for cleaning

In [46]:
loans_applicants = pd.read_csv("Loan_Applicants.csv", sep="|")
loans_applicants.shape

(22, 10)

In [47]:
loans_applicants[loans_applicants.isna().any(axis=1)]

,No.,Name of Group/Name of Business/Company,Ward,Zone,Contact Person (Name),Contact Person (Phone No.),Type of Business,Sector,Loan Amount Requested,Remarks
0,No.,Name of Group/ Name of Business/Company,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,Ward,Zone,Contact Person (Name),Contact Person (Phone No.),Type of Business,Sector,Loan Amount Requested,Remarks
7,6,CRISYA FISHING & BAKERY,KKOMA,KKOMA,Mino Matendele,NaN,NaN,NaN,NaN,NaN
14,13,Jasperd Mazambani General Dealers,Syampande,Syampande,Olipa Mapulanga,NaN,NaN,NaN,NaN,NaN
21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"3,599,000.00",NaN


In [48]:
loans_applicants.head()

,No.,Name of Group/Name of Business/Company,Ward,Zone,Contact Person (Name),Contact Person (Phone No.),Type of Business,Sector,Loan Amount Requested,Remarks
0,No.,Name of Group/ Name of Business/Company,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,Ward,Zone,Contact Person (Name),Contact Person (Phone No.),Type of Business,Sector,Loan Amount Requested,Remarks
2,1,MACHEYA HABUNGA GENERAL DEALERS,KKOLE,KKOLE,Gilbert Mucheya,979521282,PROCUREMENT OF TRUCK,TRANSPORT,"200,000.00","Certificate of Registration, TPIN Certificate,..."
3,2,MUTEMENA BUSINESS VENTURES,CHAAMWE,NYANGA,Maggie Choonda,951573083,IRRIGATION PROJECT,AGRICULTURE,"100,000.00","Certificate of Registration, TPIN Certificate,..."
4,3,BENISTER HAMWIINGA ENTERPRISE,CHAAMWE,NYANGA,Benister Hamwiinga,0,IRRIGATION PROJECT,AGRICULTURE,"100,000.00","Certificate of Registration, TPIN Certificate,..."


In [49]:
print(loans_applicants["Loan Amount Requested"].dtype) 

str


## Cleaning Loan Applicants CSV file

In [50]:
loans_applicants = loans_applicants.drop("No.", axis=1)
loans_applicants = loans_applicants.drop(index=[0,1]).reset_index(drop=True)

In [51]:
loans_applicants = loans_applicants.dropna(ignore_index=True)

In [52]:
loans_applicants = loans_applicants.apply(lambda col: col.str.lower())

In [53]:
loans_applicants["Loan Amount Requested"] = pd.to_numeric(loans_applicants["Loan Amount Requested"].str.replace(',', ''), errors='coerce')

In [54]:
print(loans_applicants["Loan Amount Requested"].dtype)

float64


In [55]:
loans_applicants.columns = loans_applicants.columns.str.strip().str.lower().str.replace(' ', '_')   #column nomalization

In [56]:
loans_applicants.head(10)

,name_of_group/name_of_business/company,ward,zone,contact_person_(name),contact_person_(phone_no.),type_of_business,sector,loan_amount_requested,remarks
0,macheya habunga general dealers,kkole,kkole,gilbert mucheya,979521282,procurement of truck,transport,200000.0,"certificate of registration, tpin certificate,..."
1,mutemena business ventures,chaamwe,nyanga,maggie choonda,951573083,irrigation project,agriculture,100000.0,"certificate of registration, tpin certificate,..."
2,benister hamwiinga enterprise,chaamwe,nyanga,benister hamwiinga,0,irrigation project,agriculture,100000.0,"certificate of registration, tpin certificate,..."
3,hamunyewu effort general dealers,kkota kkota,mabula,effort hamunyewu,950177241,irrigation project,agriculture,200000.0,"certificate of registration, tpin certificate,..."
4,gilbert siakumbo fishing enterprise,syambabala,siabwengo,dorcas simayoba,0973078667,kapenta fishing,fisheries,200000.0,"certificate of registration, tpin certificate,..."
5,wesley siazibone general dealers,kkoma,malilasuntwe,peggy sialwiinga,979971307,procurement of a mini bus,transport,200000.0,"certificate of registration, tpin certificate,..."
6,matiibi multi-purpose cooperative,chisanga,siabbamba,chrisborn mutonta,979614332,procurement of truck,transport,200000.0,"certificate of registration, tpin certificate,..."
7,nithel enterprise,lukonde,gwembe,best moonde,972393955,block making machine project,construction,200000.0,"certificate of registration, tpin certificate,..."
8,nkalanga business venture,sompani,sompani,enoventer chikuni,954633793,poultry project,livestock,200000.0,"certificate of registration, tpin certificate,..."
9,promise sepiso general dealers,makuyu,katete,promise sepiso haatyoka,0979827464,irrigation,agriculture,200000.0,"certificate of registration, tpin certificate,..."


### Saving cleaned loans_applicants to csv

In [57]:
loans_applicants.to_csv(
    "cleand_loans_applicants.csv",
    sep="|",
    index=False
)

# Cleaning Empowerment Grants CSV file

### Inspecting the Empowerment Grants CSV file

In [58]:
empowerment_grants = pd.read_csv("empowerment_grants.csv", sep="|")

In [59]:
empowerment_grants.shape

(97, 5)

In [60]:
empowerment_grants[empowerment_grants.isna().any(axis=1)]

,S/N,Name Of Applicant,Ward,Project Title,Approved Amount
17,NaN,Hatyelele Development Group,Luumbo,Piggery Project,"20,000"
32,NaN,Dulu Club,Chaamwe,Beef/Cattle Production,"20,000"
67,NaN,NaN,NaN,NaN,"40,000.00"
96,TOTAL,NaN,NaN,NaN,"2,259,375.00"


In [61]:
empowerment_grants.head(10)

,S/N,Name Of Applicant,Ward,Project Title,Approved Amount
0,S/N,Name Of Applicant,Ward,Project Title,Approved amount
1,1.,Machembele Women's Club,Lukonde,Goat Rearing,"15,000"
2,2.,Hakazembwe Multi-Purpose Cooperative,Lukonde,Goat Rearing,"15,000"
3,3.,Bulimi Bubotu Women's Cooperative,Lukonde,Chicken Rearing Project,"10,000"
4,4.,Tutante Club,Lukonde,Goat Rearing,"15,000"
5,5.,Chuunga Club,Lukonde,Goat Rearing,"15,000"
6,6.,Katulanga Club,Syampande,Goat Rearing and Butchery,"15,000"
7,7.,Kasika Club,Syampande,Piggery Project,"20,000"
8,8.,Kabbudula Club,Syampande,Goat Project,"15,000"
9,9.,Ngombe Buka Club,Syampande,Bee Keeping Project,"15,000"


In [62]:
print(empowerment_grants["Approved Amount"].dtype) 

str


### Cleaning Empowerment Grants CSV file

In [63]:
empowerment_grants = empowerment_grants.drop("S/N", axis=1)

In [64]:
empowerment_grants = empowerment_grants.drop(index=0).reset_index(drop=True)

In [65]:
empowerment_grants = empowerment_grants.apply(lambda col: col.str.lower())

In [66]:
empowerment_grants["Approved Amount"] = pd.to_numeric(empowerment_grants["Approved Amount"].str.replace(',', ''), errors='coerce')

In [67]:
print(empowerment_grants["Approved Amount"].dtype) 

float64


In [68]:
empowerment_grants = empowerment_grants.dropna(ignore_index=True)

In [69]:
empowerment_grants[empowerment_grants.isna().any(axis=1)]

,Name Of Applicant,Ward,Project Title,Approved Amount


In [70]:
empowerment_grants.columns = empowerment_grants.columns.str.strip().str.lower().str.replace(' ', '_')   #column nomalization

In [71]:
empowerment_grants.head(15)

,name_of_applicant,ward,project_title,approved_amount
0,machembele women's club,lukonde,goat rearing,15000.0
1,hakazembwe multi-purpose cooperative,lukonde,goat rearing,15000.0
2,bulimi bubotu women's cooperative,lukonde,chicken rearing project,10000.0
3,tutante club,lukonde,goat rearing,15000.0
4,chuunga club,lukonde,goat rearing,15000.0
5,katulanga club,syampande,goat rearing and butchery,15000.0
6,kasika club,syampande,piggery project,20000.0
7,kabbudula club,syampande,goat project,15000.0
8,ngombe buka club,syampande,bee keeping project,15000.0
9,mutombe club,syampande,pig project and butchery,20000.0


### Saving cleaned empowerment_grants to csv

In [72]:
empowerment_grants.to_csv(
    "cleaned_empowerment_grants.csv",
    sep="|",
    index=False
)

# Cleaning Community Projects CSV file

### Inspecting the Community Projects CSV file

In [73]:
community_projects = pd.read_csv("community_projects.csv", sep="|")

In [74]:
community_projects.shape

(8, 8)

In [75]:
community_projects[community_projects.isna().any(axis=1)]

,S/N,Project Name,Project Description,Sector,Project,Ward,Zone,Location
5,6.,Rural Electrification,Electrification of Kalama Primary School,Power Supply,Construction,Jumbo - Sompani Ward,NaN,Kalama Primary School
7,"TOTAL K18,304,493.20",NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [76]:
community_projects

,S/N,Project Name,Project Description,Sector,Project,Ward,Zone,Location
0,S/N 1.,Project Name Construction of a Health Post & A...,Project Description Construction of a Health P...,Sector Health,Project Construction,Ward Kkota- Kkota,Zone Mabula,Location Mabula
1,2.,Construction of a 1X3 Classroom Block at Bbond...,Construction of a 1X3 Classroom Block,Education,Construction,Bbondo,Bbondo,Bbondo
2,3.,Construction of a Health Post & a Maternity An...,Construction of a Health Post & Maternity Annex,Health,Construction,Jongola,Jongola,Jongola Area
3,4.,Construction of a 1X3 Classroom Block at Mukon...,Construction of a 1X3 Classroom Block,Education,Construction,Chaamwe,Mukono,Mukuno Community School
4,5.,Maintenance of Roads in the District,Maintenance of Roads & Servicing of Equipment,Road Infrastructure,Construction,Gwembe District,Gwembe District,Gwembe District
5,6.,Rural Electrification,Electrification of Kalama Primary School,Power Supply,Construction,Jumbo - Sompani Ward,NaN,Kalama Primary School
6,7.,Procurement of a Drilling Rig & a Siting Machine,Procurement of a Drilling Rig & a Siting Machine,Water,Procurement,Gwembe District,Gwembe District,Gwembe District
7,"TOTAL K18,304,493.20",NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Cleaning Community Projects CSV file

In [77]:
community_projects = community_projects.drop("S/N", axis=1)

In [78]:
community_projects = community_projects.drop(index=0).reset_index(drop=True)

In [79]:
community_projects = community_projects.dropna(ignore_index=True)

In [80]:
community_projects = community_projects.apply(lambda col: col.str.lower())

In [81]:
loans_applicants.columns = loans_applicants.columns.str.strip().str.lower().str.replace(' ', '_')   #column nomalization

In [82]:
loans_applicants.head()

,name_of_group/name_of_business/company,ward,zone,contact_person_(name),contact_person_(phone_no.),type_of_business,sector,loan_amount_requested,remarks
0,macheya habunga general dealers,kkole,kkole,gilbert mucheya,979521282,procurement of truck,transport,200000.0,"certificate of registration, tpin certificate,..."
1,mutemena business ventures,chaamwe,nyanga,maggie choonda,951573083,irrigation project,agriculture,100000.0,"certificate of registration, tpin certificate,..."
2,benister hamwiinga enterprise,chaamwe,nyanga,benister hamwiinga,0,irrigation project,agriculture,100000.0,"certificate of registration, tpin certificate,..."
3,hamunyewu effort general dealers,kkota kkota,mabula,effort hamunyewu,950177241,irrigation project,agriculture,200000.0,"certificate of registration, tpin certificate,..."
4,gilbert siakumbo fishing enterprise,syambabala,siabwengo,dorcas simayoba,0973078667,kapenta fishing,fisheries,200000.0,"certificate of registration, tpin certificate,..."


### Saving cleaned community_projects to csv

In [83]:
loans_applicants.to_csv(
    "cleaned_community_projects.csv",
    sep="|",
    index=False
)

# Cleaning School Bursaries CSV file

### Inspecting the School Bursaries CSV file

In [84]:
school_bursaries = pd.read_csv("school_bursaries.csv", sep="|")

In [85]:
school_bursaries.shape

(50, 9)

In [86]:
school_bursaries[school_bursaries.isna().any(axis=1)]

,S/N,Name of Pupil,Ward,Sex (M/F),Grade,Name of School,School Location (District),Termly Fees,Annual Fees
9,NaN,NaN,NaN,NaN,NaN,NaN,Gwembe,NaN,NaN
17,NaN,Hang'andu Markson,Makuyu,M,10,Munyumbwe Secondary School,Gwembe,"1,000","3,000"
26,NaN,NaN,NaN,NaN,NaN,NaN,Munyumb we,NaN,NaN
35,NaN,Hanyulu Osylder,NaN,NaN,8,NaN,NaN,NaN,NaN
44,NaN,Muyaule Elisha Elias,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45,45,Simaswaana Chrispine,Chibuwe,NaN,9,Chipepo Secondary School,Chipepo,"1,000","3,000"
49,Total,NaN,NaN,NaN,NaN,NaN,NaN,"48,000.00","144,000.00"


In [87]:
school_bursaries

,S/N,Name of Pupil,Ward,Sex (M/F),Grade,Name of School,School Location (District),Termly Fees,Annual Fees
0,S/N,Name of Pupil,Ward,Sex (M/ F),Grade,Name of School,School Location (District),Termly Fees,Annual Fees
1,1,Valarie Hapunda,Chaamwe,F,8,Kaumba Secondary School,Monze,"1,000","3,000"
2,2,Agripa Kalyata,Kkota Kkota,M,8,Munyumbwe Secondary School,Gwembe,"1,000","3,000"
3,3,Stanford Hamunyeu,Kkota Kkota,M,8,Chipepo Secondary School,Gwembe,"1,000","3,000"
4,4,Reuben Mwanza,Kkota Kkota,M,8,Chipepo Secondary School,Gwembe,"1,000","3,000"
5,5,Brian Jeke,Kkota Kkota,M,8,Chipepo Secondary School,Gwembe,"1,000","3,000"
6,6,Clive Hachitema,Kkota Kkota,M,8,Chipepo Secondary School,Gwembe,"1,000","3,000"
7,7,Talent Mwiinde,Kkota Kkota,M,11,Chipepo Secondary School,Gwembe,"1,000","3,000"
8,8,Brave Hamabeku,Kkota Kkota,M,11,Chipepo Secondary School,Gwembe,"1,000","3,000"
9,NaN,NaN,NaN,NaN,NaN,NaN,Gwembe,NaN,NaN


### Cleaning Community Projects CSV file

In [88]:
school_bursaries = school_bursaries.drop("S/N", axis=1)

In [89]:
school_bursaries = school_bursaries.drop(index=0).reset_index(drop=True)

In [90]:
school_bursaries[school_bursaries.isna().any(axis=1)]

,Name of Pupil,Ward,Sex (M/F),Grade,Name of School,School Location (District),Termly Fees,Annual Fees
8,NaN,NaN,NaN,NaN,NaN,Gwembe,NaN,NaN
25,NaN,NaN,NaN,NaN,NaN,Munyumb we,NaN,NaN
34,Hanyulu Osylder,NaN,NaN,8,NaN,NaN,NaN,NaN
43,Muyaule Elisha Elias,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44,Simaswaana Chrispine,Chibuwe,NaN,9,Chipepo Secondary School,Chipepo,"1,000","3,000"
48,NaN,NaN,NaN,NaN,NaN,NaN,"48,000.00","144,000.00"


In [91]:
school_bursaries.loc[44, 'Sex (M/F)'] = 'M'

In [92]:
school_bursaries[school_bursaries.isna().any(axis=1)]

,Name of Pupil,Ward,Sex (M/F),Grade,Name of School,School Location (District),Termly Fees,Annual Fees
8,NaN,NaN,NaN,NaN,NaN,Gwembe,NaN,NaN
25,NaN,NaN,NaN,NaN,NaN,Munyumb we,NaN,NaN
34,Hanyulu Osylder,NaN,NaN,8,NaN,NaN,NaN,NaN
43,Muyaule Elisha Elias,NaN,NaN,NaN,NaN,NaN,NaN,NaN
48,NaN,NaN,NaN,NaN,NaN,NaN,"48,000.00","144,000.00"


In [93]:
school_bursaries = school_bursaries.dropna(ignore_index=True)

In [94]:
school_bursaries[school_bursaries.isna().any(axis=1)]

,Name of Pupil,Ward,Sex (M/F),Grade,Name of School,School Location (District),Termly Fees,Annual Fees


In [95]:
school_bursaries = school_bursaries.apply(lambda col: col.str.lower())

In [96]:
school_bursaries.columns = school_bursaries.columns.str.strip().str.lower().str.replace(' ', '_')   #column nomalization

In [97]:
school_bursaries.head()

,name_of_pupil,ward,sex_(m/f),grade,name_of_school,school_location_(district),termly_fees,annual_fees
0,valarie hapunda,chaamwe,f,8,kaumba secondary school,monze,"1,000","3,000"
1,agripa kalyata,kkota kkota,m,8,munyumbwe secondary school,gwembe,"1,000","3,000"
2,stanford hamunyeu,kkota kkota,m,8,chipepo secondary school,gwembe,"1,000","3,000"
3,reuben mwanza,kkota kkota,m,8,chipepo secondary school,gwembe,"1,000","3,000"
4,brian jeke,kkota kkota,m,8,chipepo secondary school,gwembe,"1,000","3,000"


In [98]:
print(school_bursaries["termly_fees"].dtype) 

str


In [99]:
print(school_bursaries["annual_fees"].dtype) 

str


In [100]:
school_bursaries["termly_fees"] = pd.to_numeric(school_bursaries["termly_fees"].str.replace(',', ''), errors='coerce')

In [101]:
print(school_bursaries["termly_fees"].dtype) 

int64


In [102]:
school_bursaries["annual_fees"] = pd.to_numeric(school_bursaries["annual_fees"].str.replace(',', ''), errors='coerce')

In [103]:
print(school_bursaries["annual_fees"].dtype) 

int64


In [104]:
school_bursaries.head(10)

,name_of_pupil,ward,sex_(m/f),grade,name_of_school,school_location_(district),termly_fees,annual_fees
0,valarie hapunda,chaamwe,f,8,kaumba secondary school,monze,1000,3000
1,agripa kalyata,kkota kkota,m,8,munyumbwe secondary school,gwembe,1000,3000
2,stanford hamunyeu,kkota kkota,m,8,chipepo secondary school,gwembe,1000,3000
3,reuben mwanza,kkota kkota,m,8,chipepo secondary school,gwembe,1000,3000
4,brian jeke,kkota kkota,m,8,chipepo secondary school,gwembe,1000,3000
5,clive hachitema,kkota kkota,m,8,chipepo secondary school,gwembe,1000,3000
6,talent mwiinde,kkota kkota,m,11,chipepo secondary school,gwembe,1000,3000
7,brave hamabeku,kkota kkota,m,11,chipepo secondary school,gwembe,1000,3000
8,conelious hamunyeu,kkota kkota,m,11,chipepo secondary school,gwembe,1000,3000
9,augustine syagonyo,chisanga,m,10,namwala secondary school,namwala,1000,3000


### Saving cleaned community_projects to csv

In [105]:
loans_applicants.to_csv(
    "cleaned_school_bursaries.csv",
    sep="|",
    index=False
)

# Cleaning Skills Development Bursaries CSV file

### Inspecting the Skills Development Bursaries CSV file

In [106]:
import pandas as pd
skills_development_bursaries = pd.read_csv("skills_development_bursaries.csv", sep="|")

In [107]:
skills_development_bursaries.shape

(275, 13)

In [108]:
skills_development_bursaries[skills_development_bursaries.isna().any(axis=1)]

,S/N,Name of Student,Ward,Sex (M/F),Date of Birth,Name of Skill/Programme,Level of Skill,Programme Duration (No. of Months),Training Institute (TEVET/ZNS),Term One,Term Two,Term Three,Annual Tuition Fees
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"13,200",0,0,"13,200.00"
12,12,Stermon Muzovwa,Syambabala,M,09-11-03,Automotive Engineering,Certificate,24,LIBES,"27,940",NaN,0,"27,940.00"
15,15,Vera Mubita,Syambabala,F,10-05-04,Electrical Technology,Certificate,3,LIBES,"9,530",0,NaN,"9,530.00"
20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"9,530",0,0,"9,530.00"
38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,"9,530",0,0,"9,530.00"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
258,240,Syamusika Nchimunya,Chibuwe,M,22/1/1996,Food production,Certificate,1 year,Livingstone Institute of Business and Engineering,NaN,NaN,NaN,NaN
265,NaN,Siampondo Fredrick,NaN,M,NaN,NaN,NaN,1 year,NaN,"9,530",0,0,"9,530.00"
266,248,Nyambe Josephine,Chibuwe,M,12-05-01,Food production,Certificate,3 Months,Livingstone Institute of Business and Engineering,"9,530",NaN,0,"9,530.00"
268,250,Musabi Newton,Chibuwe,M,30/05/1997,Heavy Equipment,Certificate,3 years,Livingstone Institute of Business and Engineering,"9,530",0,NaN,"28,590.00"


In [109]:
skills_development_bursaries

,S/N,Name of Student,Ward,Sex (M/F),Date of Birth,Name of Skill/Programme,Level of Skill,Programme Duration (No. of Months),Training Institute (TEVET/ZNS),Term One,Term Two,Term Three,Annual Tuition Fees
0,S/N,Name of Student,WARD (,SEX M/F),Date of Birth,Name of Skill/Programm e,Level of Skill e.g Certificat e/diploma,Progra mme Duratio n (No. of Months),Training Institute (TEVET/ZNS),Term One,Term Two,Ter m Thr ee,Annual Tuition fees
1,1,Kachalo Cheembo,Chaamwe,M,09-01-01,General Agriculture,Diploma,36,Zambia College Agriculture,"4,880",0,"4,88 0","9,760.00"
2,2,Matany Muulu,Chaamwe,M,04-03-93,Heavy Equipment Engineering,Diploma,36,LIBES,"27,240",0,0,"27,240.00"
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"13,200",0,0,"13,200.00"
4,4,Before Nzala,Chaamwe,M,04-04-02,Auto-Mechanics,Diploma,24,St. Mawaggali Trades Training Institution,"11,900","11,000","13,6 00","36,500.00"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,252,Mapulanga Victor,Chibuwe,M,06-01-04,Electrical Engineering,Certificate,2 years,Livingstone Institute of Business and Engineering,"27,160",0,0,"27,160.00"
271,253,Siamatondo Bentry,Chibuwe,M,25/06/2003,Automotive Mechanics Trade,Certificate,3 Months,Livingstone Institute of Business and Engineering,"9,530",0,0,"9,530.00"
272,254,Siamatondo Bright,Chibuwe,M,27/08/2000,Carpentry and Joinery,Certificate,4 Months,Livingstone Institute of Business and Engineering,"9,530",0,0,"9,530.00"
273,255,Hamoobola Japhet,Jongola,M,02-02-92,Heavy Duty Truck Driving,Certificate,1 Month,ITC,"23,510",0,0,"23,510.00"


### Cleaning Skills Development Bursaries CSV file

In [110]:
skills_development_bursaries = skills_development_bursaries.drop("S/N", axis=1)

In [111]:
skills_development_bursaries = skills_development_bursaries.drop(index=0).reset_index(drop=True)

In [112]:
skills_development_bursaries = skills_development_bursaries.apply(lambda col: col.str.lower())

In [113]:
skills_development_bursaries.columns = skills_development_bursaries.columns.str.strip().str.lower().str.replace(' ', '_')   #column nomalization

In [114]:
skills_development_bursaries = skills_development_bursaries.dropna(subset=["name_of_skill/programme"], ignore_index=True)

In [115]:
skills_development_bursaries[skills_development_bursaries.isna().any(axis=1)]

,name_of_student,ward,sex_(m/f),date_of_birth,name_of_skill/programme,level_of_skill,programme_duration_(no._of_months),training_institute_(tevet/zns),term_one,term_two,term_three,annual_tuition_fees
10,stermon muzovwa,syambabala,m,09-11-03,automotive engineering,certificate,24,libes,"27,940",NaN,0,"27,940.00"
13,vera mubita,syambabala,f,10-05-04,electrical technology,certificate,3,libes,"9,530",0,NaN,"9,530.00"
49,mapenzi mwaangwa,NaN,m,NaN,fire engineering,NaN,36,eden university,"15,160","13,960","13,9 60","43,080.00"
50,NaN,NaN,NaN,NaN,& rescue,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,NaN,NaN,NaN,NaN,management,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
219,NaN,NaN,NaN,NaN,electrical and electronics,NaN,3 years,NaN,"16,300","16,300",0,"32,600.00"
237,mutempa enny,NaN,f,NaN,general agriculture,NaN,1 years,kasiya college,"5,485",0,0,"16,455.00"
246,syamusika nchimunya,chibuwe,m,22/1/1996,food production,certificate,1 year,livingstone institute of business and engineering,NaN,NaN,NaN,NaN
253,nyambe josephine,chibuwe,m,12-05-01,food production,certificate,3 months,livingstone institute of business and engineering,"9,530",NaN,0,"9,530.00"


In [116]:
skills_development_bursaries = skills_development_bursaries.dropna(subset=["name_of_student", "ward", "date_of_birth"], ignore_index=True)

In [117]:
skills_development_bursaries = skills_development_bursaries.dropna(subset=["term_one", "term_two", "term_three"], ignore_index=True)

In [118]:
skills_development_bursaries[skills_development_bursaries.isna().any(axis=1)]

,name_of_student,ward,sex_(m/f),date_of_birth,name_of_skill/programme,level_of_skill,programme_duration_(no._of_months),training_institute_(tevet/zns),term_one,term_two,term_three,annual_tuition_fees


In [119]:
skills_development_bursaries.head(20)

,name_of_student,ward,sex_(m/f),date_of_birth,name_of_skill/programme,level_of_skill,programme_duration_(no._of_months),training_institute_(tevet/zns),term_one,term_two,term_three,annual_tuition_fees
0,kachalo cheembo,chaamwe,m,09-01-01,general agriculture,diploma,36,zambia college agriculture,"4,880",0,"4,88 0","9,760.00"
1,matany muulu,chaamwe,m,04-03-93,heavy equipment engineering,diploma,36,libes,"27,240",0,0,"27,240.00"
2,before nzala,chaamwe,m,04-04-02,auto-mechanics,diploma,24,st. mawaggali trades training institution,"11,900","11,000","13,6 00","36,500.00"
3,rhodah simunda,kkota kkota,f,09-03-99,food production,certificate,3,libes,"9,530",0,0,"9,530.00"
4,rita muleya,kkota kkota,f,26-10-06,fashion design & textile,certificate,3,libes,"9,530",0,0,"9,530.00"
5,bright hamunyewu,kkota kkota,m,01-01-92,metal fabrication,certificate,3,kasiya college,"5,400",0,0,"5,400.00"
6,friday simunyama,kkota kkota,m,01-02-90,general agriculture,certificate,12,kasiya college,"5,400","5,400","5,40 0","16,200.00"
7,vincent siamaamba,kkota kkota,m,08-08-04,metal fabrication,certificate,3,kasiya college,"5,400",0,0,"5,400.00"
8,lucky sialenga,syambabala,m,15-11-95,bricklaying & plastering,certificate,3,libes,"9,530",0,0,"9,530.00"
9,trade muzungu,syambabala,m,22-09-03,computer studies,certificate,3,libes,"9,530",0,0,"9,530.00"


### Saving cleaned skills_development_bursaries to csv

In [120]:
skills_development_bursaries.to_csv(
    "cleaned_skills_development_bursaries.csv",
    sep="|",
    index=False
)